# OmniVoice Text-to-Speech with OpenVINO™

[OmniVoice](https://github.com/k2-fsa/OmniVoice) is a state-of-the-art zero-shot text-to-speech system from Xiaomi AI Lab's next-generation Kaldi team. It supports **600+ languages**, voice cloning from a short reference clip, and voice design via natural-language attribute prompts.

Unlike typical TTS pipelines, OmniVoice is **diffusion-style**: the LLM is invoked once per timestep (32 steps by default) on the full sequence, with classifier-free guidance and confidence-driven token unmasking. There is no KV-cache to reuse — every step is a full bidirectional forward pass. The 32-step loop, sampling, and pre/post-processing all live in Python; the heavy weighted sub-models run on OpenVINO.

In this tutorial we will:
1. Convert the four weighted sub-models (Qwen3 LLM, HiggsAudio v2 encoder/decoder, Whisper-large-v3-turbo) to OpenVINO IR
2. Optionally quantize the LLM to INT8 with NNCF
3. Run inference on CPU and GPU
4. Compare quality against the original PyTorch model on CPU
5. Launch an interactive Gradio demo

#### Table of contents
- [Prerequisites](#Prerequisites)
- [Convert OmniVoice to OpenVINO](#Convert-OmniVoice-to-OpenVINO)
- [Select Inference Devices](#Select-Inference-Devices)
- [Run Inference](#Run-Inference)
- [Compare with PyTorch CPU baseline](#Compare-with-PyTorch-CPU-baseline)
- [Interactive Demo](#Interactive-Demo)

⚠️ **EXPERIMENTAL NOTEBOOK** — OmniVoice has not been fully validated with OpenVINO.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/omnivoice/omnivoice.ipynb" />

## Prerequisites
[back to top ⬆️](#Table-of-contents)

In [ ]:
import requests
from pathlib import Path

for fname in ("notebook_utils.py", "cmd_helper.py", "pip_helper.py"):
    if not Path(fname).exists():
        r = requests.get(
            f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{fname}"
        )
        Path(fname).write_text(r.text)

from notebook_utils import collect_telemetry

collect_telemetry("omnivoice.ipynb")

### Install dependencies
[back to top ⬆️](#Table-of-contents)

In [ ]:
from pip_helper import pip_install

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch>=2.4",
    "torchaudio>=2.4",
    "transformers>=5.3",
    "openvino>=2025.4",
    "nncf",
    "optimum-intel[openvino]",
    "omnivoice",
    "gradio>=4.0",
    "soundfile",
    "librosa",
    "huggingface_hub",
)

## Convert OmniVoice to OpenVINO
[back to top ⬆️](#Table-of-contents)

The conversion produces four IR models:

1. `openvino_llm_model.xml` — Qwen3-0.6B + audio embedding + `audio_heads` projection, fused into one graph (INT8 optional)
2. `openvino_audio_encoder.xml` — HiggsAudio v2 encoder (waveform → 8-codebook tokens)
3. `openvino_audio_decoder.xml` — HiggsAudio v2 decoder (codes → waveform)
4. `whisper/` — Whisper-large-v3-turbo, used only when reference audio is supplied without a transcript

Conversion is per-sub-model and is skipped if the IR already exists.

In [ ]:
import ipywidgets as widgets

to_quantize = widgets.Checkbox(
    value=True,
    description="INT8 LLM weights",
    style={"description_width": "initial"},
)

convert_whisper = widgets.Checkbox(
    value=True,
    description="Convert Whisper for ref-audio auto-transcription (~3GB download)",
    style={"description_width": "initial"},
)

display(to_quantize, convert_whisper)

In [ ]:
from pathlib import Path
import nncf

from omnivoice_helper import convert_omnivoice

MODEL_ID = "k2-fsa/OmniVoice"
OV_MODEL_DIR = Path("ov_model_int8" if to_quantize.value else "ov_model")

convert_omnivoice(
    model_id=MODEL_ID,
    output_dir=OV_MODEL_DIR,
    llm_quantization_config={"mode": nncf.CompressWeightsMode.INT8_SYM} if to_quantize.value else None,
    convert_whisper=convert_whisper.value,
    asr_model_id="openai/whisper-large-v3-turbo",
)

## Select Inference Devices
[back to top ⬆️](#Table-of-contents)

Each sub-model can use a different device. NPU is excluded because OmniVoice's diffusion graph requires fully dynamic shapes.

In [ ]:
from notebook_utils import device_widget

llm_device = device_widget("CPU", exclude=["NPU"], description="LLM:")
audio_device = device_widget("CPU", exclude=["NPU"], description="Audio tokenizer:")
asr_device = device_widget("CPU", exclude=["NPU"], description="Whisper ASR:")

display(llm_device, audio_device, asr_device)

In [ ]:
from omnivoice_helper import OVOmniVoice

ov_model = OVOmniVoice.from_pretrained(
    OV_MODEL_DIR,
    llm_device=llm_device.value,
    audio_device=audio_device.value,
    asr_device=asr_device.value,
)
print("✓ OVOmniVoice ready. sampling_rate =", ov_model.sampling_rate)

## Run Inference
[back to top ⬆️](#Table-of-contents)

OmniVoice supports three modes:

- **Voice Design** — describe the voice with attributes (gender, age, pitch, accent, style)
- **Voice Clone** — supply reference audio (and optionally its transcript)
- **Auto** — supply nothing; the model picks a voice

In [ ]:
import time
import IPython.display as ipd

text_en = "Hello! Welcome to the OmniVoice demo with OpenVINO acceleration."

t0 = time.time()
audios = ov_model.generate(
    text=text_en,
    language="English",
    instruct="female, young adult, moderate pitch",
)
elapsed = time.time() - t0

wav = audios[0]
duration = len(wav) / ov_model.sampling_rate
print(f"Inference: {elapsed:.2f}s | Audio: {duration:.2f}s | RTF: {elapsed/duration:.2f}")
ipd.display(ipd.Audio(wav, rate=ov_model.sampling_rate))

In [ ]:
text_zh = "你好，欢迎使用 OmniVoice 演示。这是 OpenVINO 加速后的中文语音合成。"

t0 = time.time()
audios = ov_model.generate(
    text=text_zh,
    language="Chinese",
    instruct="男，青年",
)
elapsed = time.time() - t0
wav = audios[0]
duration = len(wav) / ov_model.sampling_rate
print(f"Inference: {elapsed:.2f}s | Audio: {duration:.2f}s | RTF: {elapsed/duration:.2f}")
ipd.display(ipd.Audio(wav, rate=ov_model.sampling_rate))

## Compare with PyTorch CPU baseline
[back to top ⬆️](#Table-of-contents)

This is gated behind a checkbox because loading the original PyTorch model uses ~3 GB of memory and takes a few minutes. Because OmniVoice's loop uses Gumbel-stochastic sampling, OV and PyTorch outputs are not bit-identical even with a fixed seed (FP16 vs FP32, different sampling kernels), but they should be perceptually close and have similar RMS / log-mel statistics.

In [ ]:
do_parity_check = widgets.Checkbox(value=False, description="Run PyTorch CPU parity check")
do_parity_check

In [ ]:
if do_parity_check.value:
    import numpy as np
    import torch
    from omnivoice import OmniVoice, OmniVoiceGenerationConfig

    print("Loading PyTorch OmniVoice on CPU (FP32)...")
    pt_model = OmniVoice.from_pretrained(MODEL_ID, dtype=torch.float32, device_map="cpu")
    pt_model.eval()

    parity_text = "OpenVINO and PyTorch comparison."
    cfg = OmniVoiceGenerationConfig(num_step=16)

    torch.manual_seed(0)
    pt_wav = pt_model.generate(text=parity_text, language="English",
                               instruct="female, young adult",
                               generation_config=cfg)[0]
    torch.manual_seed(0)
    ov_wav = ov_model.generate(text=parity_text, language="English",
                               instruct="female, young adult",
                               generation_config=cfg)[0]

    pt_rms = float(np.sqrt(np.mean(pt_wav.astype(np.float64) ** 2)))
    ov_rms = float(np.sqrt(np.mean(ov_wav.astype(np.float64) ** 2)))
    print(f"PyTorch: {len(pt_wav)/pt_model.sampling_rate:.2f}s, RMS={pt_rms:.4f}")
    print(f"OpenVINO: {len(ov_wav)/ov_model.sampling_rate:.2f}s, RMS={ov_rms:.4f}")
    print(f"RMS ratio: {ov_rms/pt_rms:.3f} (close to 1.0 expected)")
    print("\nPyTorch:")
    ipd.display(ipd.Audio(pt_wav, rate=pt_model.sampling_rate))
    print("\nOpenVINO:")
    ipd.display(ipd.Audio(ov_wav, rate=ov_model.sampling_rate))
    del pt_model

## Interactive Demo
[back to top ⬆️](#Table-of-contents)

Two tabs (mirroring the [HuggingFace Space](https://huggingface.co/spaces/k2-fsa/OmniVoice)):

- **Voice Clone** — upload a reference audio (3–10 s recommended) and synthesize new text in the same voice
- **Voice Design** — select speaker attributes from dropdowns to generate a custom voice

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_model)

try:
    demo.queue().launch(debug=True)
except Exception:
    demo.queue().launch(debug=True, share=True)
# Use demo.launch(server_name=..., server_port=...) to bind to a remote address.